In [25]:
crypto_long_name = 'Bitcoin' # 'Bitcoin' 'Ethereum'  'Test'
crypto_abbrev = 'btc' # 'btc' 'eth'  'tst'

expiry_date = "20260618"  #"20260424"

In [26]:
from datetime import datetime

import xlwings as xw
from Class_xlWings import *
xls = xlWings()

In [27]:
wb_ratios = xw.Book("2026 Crypto ETF Ratios.xlsx")
wb_db = xw.Book("2026 Crypto Products Database.xlsx")

In [28]:
sht = wb_ratios.sheets[crypto_abbrev + " ratios"]

tbl = sht.tables[crypto_abbrev + "_ratios"]
tbl_range = tbl.range
    
ratios_df = tbl_range.options(pd.DataFrame, index=False).value

In [29]:
sht = wb_db.sheets[crypto_abbrev]

table_name = crypto_abbrev + "_true_false_table"
tbl = sht.tables[table_name]
tbl_range = tbl.range
    
eligible_df = tbl_range.options(pd.DataFrame, index=False).value
eligible_df = eligible_df[eligible_df['TRUE/FALSE'] == True]

In [30]:
dict_of_dfs = {}

for symbol in eligible_df["my_name"]:
    sht = wb_db.sheets[symbol + " Options"]
    
    symbol = symbol.lower()
    table_name = symbol + "_options"
    tbl = sht.tables[table_name]
    tbl_range = tbl.range

    df = tbl_range.options(pd.DataFrame, index=False).value

    dict_of_dfs[symbol] = df

opts_df = pd.concat(
    [df.assign(symbol=symbol) for symbol, df in dict_of_dfs.items()],
    ignore_index=True
)

opts_df = opts_df.drop(columns=["symbol"])

In [31]:
expiry_date = datetime.strptime(expiry_date, "%Y%m%d").date()
ratios_df["Date"] = pd.to_datetime(ratios_df["Date"], format="%Y-%m-%d").dt.date
opts_df['expiry'] = pd.to_datetime(opts_df["expiry"], format="%Y%m%d").dt.date

ratios_df = ratios_df[ratios_df['Date'] == expiry_date]
opts_df = opts_df[opts_df['expiry'] == expiry_date]

In [32]:
value_map = ratios_df.iloc[0].to_dict()
opts_df["shs/btc"] = opts_df["underlying_symbol"].map(value_map)

opts_df["strike"] = opts_df["strike"].astype(float)
opts_df['btc_equiv_strike'] = opts_df['strike'] * opts_df["shs/btc"]

In [33]:
xls.printDFToXL('output.xlsx', 'Sheet1', 'A1', opts_df)